# Predicting Solar Flare Counts with Poisson and Negative Binomial GLMs

*Quinn Mackerness*

Modelling the number of common (C-class) solar flares produced by an active region
of the Sun in a 24-hour period, from the region's physical characteristics.

**Data:** UCI Solar Flare dataset (NOAA, 1978), `flare.data2`, containing 1,066 active regions,
each described by sunspot features (modified Zürich class, spot size, distribution, etc.)
along with counts of C-, M- and X-class flares produced in the following 24 hours.

**Method:** the flare count is a non-negative integer, so this is a count-regression
problem, the same family of frequency models (Poisson and negative binomial) used to model
insurance claim counts. I model the C-class counts here as the baseline, and the same method
extends to the rarer M- and X-class flares.

## Stage 1: Loading and exploring the data

In [5]:
import pandas as pd

url = 'https://raw.githubusercontent.com/doguilmak/Predicting-Solar-Flares/main/flare.data2'

# flare.data2 is space-separated, has a descriptive header line to skip, and has no column names
column_names = [
    'zurich_class', 'spot_size', 'spot_dist', 'activity', 'evolution',
    'prev_activity', 'hist_complex', 'became_complex', 'area', 'largest_spot',
    'C_flares', 'M_flares', 'X_flares'
]

df = pd.read_csv(url, sep=r'\s+', skiprows=1, names=column_names)

In [6]:
print(df.shape)
df.head()

(1066, 13)


,zurich_class,spot_size,spot_dist,activity,evolution,prev_activity,hist_complex,became_complex,area,largest_spot,C_flares,M_flares,X_flares
0,H,A,X,1,3,1,1,1,1,1,0,0,0
1,D,R,O,1,3,1,1,2,1,1,0,0,0
2,C,S,O,1,3,1,1,2,1,1,0,0,0
3,H,R,X,1,2,1,1,1,1,1,0,0,0
4,H,S,X,1,1,1,1,2,1,1,0,0,0


In [7]:
df['C_flares'].value_counts().sort_index()

C_flares
0    884
1    112
2     33
3     20
4      9
5      4
6      3
8      1
Name: count, dtype: int64

The dataset includes 1066 active solar regions, each described by categorical features of the sunspot group, such as its Zürich class, spot size and distribution. The target for this project, C_flares, is the count of common flares each region produced within a 24-hour window.

Most regions produced zero flares (884 of 1066), and the count drops off quickly after that, with very few regions at the higher values and 8 being the largest.

Because the target is a count, it can only be a whole number and can't be negative, so for example we can't have 2.7 or -3 solar flares. Linear regression isn't built for this, since it fits a straight line and predicts on a continuous scale, so on data like this where most regions have a count of 0 we'd end up with non-integer predictions. A count model (Poisson or negative binomial) is designed for exactly this kind of data, as its predictions are always non-negative whole numbers, which makes it a much better fit.

## Stage 2: Exploring the data

Before modelling, I look at the target's summary statistics as well as how the flare count varies across the predictors in order to understand the data and check for anything that needs handling.

In [13]:
df['C_flares'].describe()

count    1066.000000
mean        0.300188
std         0.835784
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         8.000000
Name: C_flares, dtype: float64

The mean C-flare count is 0.30, but the variance (std² ≈ 0.70) is more than twice that. For a Poisson distribution the two should be equal, so the data is overdispersed (the variance is larger than the mean). This is an early sign that a negative binomial model may fit better than a plain Poisson. Around 83% of regions (884 of 1066) produced no flares at all.

In [16]:
df.groupby('zurich_class')['C_flares'].mean()

zurich_class
B    0.074830
C    0.189573
D    0.418410
E    1.147368
F    0.930233
H    0.060423
Name: C_flares, dtype: float64

The average C-flare count varies a lot across the Zürich classes. The quiet classes (H and B) sit near zero, while the largest and most complex classes (E and F) average around one flare each. This fits the physics, since the Zürich class describes the size and magnetic complexity of the sunspot group, and bigger, more complex regions release more energy. It looks like a strong predictor.

In [19]:
df.groupby('spot_size')['C_flares'].mean()

spot_size
A    0.583333
H    0.296296
K    1.260870
R    0.146789
S    0.205314
X    0.075862
Name: C_flares, dtype: float64

Spot size also relates to flare count. Size K averages far more flares (about 1.26) than the rest, while X, R and S are all low. The pattern is less neatly ordered than the Zürich class, since the size codes don't follow an obvious physical order, but there's clearly some signal here, so spot size is worth keeping as a predictor.

In [22]:
df.groupby('spot_dist')['C_flares'].mean()

spot_dist
C    1.085714
I    0.695067
O    0.224319
X    0.060423
Name: C_flares, dtype: float64

Spot distribution has a clear ordered relationship with flare count. The sparse, simple distributions (X) average near zero and the average rises through O and I up to the compact, complex ones (C) at about one flare. As with the Zürich class this comes down to magnetic complexity, since more concentrated, complex fields produce more flares, so this is another strong predictor.

In [25]:
df.groupby('prev_activity')['C_flares'].mean()

prev_activity
1    0.276265
2    0.615385
3    1.120000
Name: C_flares, dtype: float64

Previous 24-hour activity is a strong predictor too. The average flare count rises steadily with the prior activity code, going from 0.28 to 0.62 to 1.12. This makes sense physically, as active regions tend to stay active, so recent flaring is a good sign of future flaring. Unlike the structural features, this one captures the region's recent behaviour rather than its physical make-up.

The remaining predictors (hist_complex, became_complex, area, largest_spot) measure the same two things I've already looked at, magnetic complexity and region size. Since they overlap with what the Zürich class and the others already capture, I left them out and built the model on the three that carry the most distinct information: Zürich class, spot distribution and previous activity.

### Stage 2 summary

The target is overdispersed (variance ≈ 0.70 against a mean of 0.30), so a negative binomial model may fit better than a plain Poisson. I test this in Stage 3.

Looking across the predictors, the flare count relates to two kinds of feature, both in ways that make physical sense. The first is magnetic complexity: the Zürich class and spot distribution both rise with flare count, since larger, more magnetically complex regions release more energy, and the Zürich class shows the strongest relationship of the two. The second is recent activity: previous 24-hour activity rises clearly with flare count, since active regions tend to stay active. That last one is almost circular, so it's worth thinking about whether it masks the structural predictors when the aim is to understand what actually drives flaring.

Spot size also rises with flare count, but it overlaps heavily with the Zürich class since both come from the same sunspot classification scheme, so I drop it to avoid the redundancy (covered in Stage 3). The remaining predictors describe these same themes, so rather than work through each one individually I left them out, leaving the three predictors above to go into the model.

## Stage 3: Frequency model

Fitting a count regression model (a Poisson GLM, then a negative binomial) to predict the C-flare count from the region's features. The overdispersion seen in Stage 2 suggests the negative binomial may fit better, so I fit both and compare them using AIC.

### A note on the predictors

As I mentioned in Stage 2, I drop `spot_size` because it overlaps heavily with `zurich_class`, since both come from the same sunspot classification scheme. Keeping both would add a lot of redundant information and make the individual effects hard to separate and interpret, without telling me anything new. The model is therefore built on `zurich_class`, `spot_dist` and `prev_activity`.

In [30]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# C() treats each predictor as categorical (unordered). prev_activity needs this 
# since it is coded as 1/2/3; without C() it would be treated as a numeric scale.
formula = 'C_flares ~ C(zurich_class) + C(spot_dist) + C(prev_activity)'

In [32]:
# Fit a Poisson GLM (the standard starting point for count data)
poisson_model = smf.glm(formula=formula, data=df, family=sm.families.Poisson()).fit()
print(poisson_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:               C_flares   No. Observations:                 1066
Model:                            GLM   Df Residuals:                     1056
Model Family:                 Poisson   Df Model:                            9
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -672.89
Date:                Tue, 30 Jun 2026   Deviance:                       917.46
Time:                        14:59:58   Pearson chi2:                 1.64e+03
No. Iterations:                     6   Pseudo R-squ. (CS):             0.2717
Covariance Type:            nonrobust                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                -2.27

In [34]:
# negative binomial: same model, allows overdispersion
nb_model = smf.glm(formula=formula, data=df, family=sm.families.NegativeBinomial(alpha=1.0)).fit()

# Compare the two models by AIC, lower is better
print(f"Poisson AIC: {poisson_model.aic:.1f}")
print(f"Negative binomial AIC: {nb_model.aic:.1f}")

Poisson AIC: 1365.8
Negative binomial AIC: 1268.5


### Reading the Poisson fit

The coefficients are on a log scale, so each one multiplies the baseline flare rate rather than adding a set number of flares. The baseline (intercept) is a quiet region producing about 0.1 flares on average.

The Zürich class is by far the strongest predictor. Its effect grows steadily across the classes and is strongly significant (p < 0.01). Compared to the baseline, class C produces roughly 2.5 times as many flares, class D about 4 times, and classes E and F around 8 to 11 times as many. Class H sits just below baseline and isn't significant, which makes sense since it is a quiet class. This matches what I saw in the exploratory analysis, where the larger, more complex regions flared more.

Spot distribution adds very little once the Zürich class is included, since none of its levels are significant on their own. Recent activity has a weaker effect, but the highest activity level is still significant (p ≈ 0.03).

The thing worth noting is the Pearson chi-squared (1640), which is well above the residual degrees of freedom (1056), a ratio of about 1.55. For a Poisson model that fits well this ratio should be close to 1, so the data is more spread out than the Poisson model allows. This is the overdispersion I expected back in Stage 2, and it is why I fit a negative binomial model next.

### Comparing Poisson and negative binomial

Both models use the same predictors, but the negative binomial allows for the extra spread that the Poisson couldn't handle. Comparing them by AIC (where lower is better), the negative binomial scores 1268 against the Poisson's 1366. That is a difference of 97, which is a big improvement.

This confirms the overdispersion I found earlier. The flare counts vary more than a Poisson model assumes, and the negative binomial deals with this much better, so it is the more appropriate model for this data.

In [38]:
# Translate the negative binomial coefficients into plain multipliers.
# Each one multiplies the baseline flare rate (exp of the coefficient).
import numpy as np

effects = np.exp(nb_model.params)
zurich_effects = effects[[i for i in effects.index if 'zurich_class' in i]]

print("Effect of Zürich class on flare rate (vs baseline class B):")
for name, mult in zurich_effects.items():
    label = name.split('T.')[1].rstrip(']')
    print(f"  Class {label}: ×{mult:.1f}")

Effect of Zürich class on flare rate (vs baseline class B):
  Class C: ×2.5
  Class D: ×4.2
  Class E: ×10.7
  Class F: ×7.9
  Class H: ×0.7


### Interpreting the model

Converting the negative binomial coefficients into multipliers shows how each Zürich class affects the flare rate compared to the quiet baseline class B. The rate goes up steadily with the size and complexity of the region: class C produces about 2.5 times the baseline rate, class D about 4 times, and classes E and F around 8 to 11 times. Class H sits just below baseline, which fits with it being a quiet class.

These effects match the patterns from the exploratory analysis and the underlying physics, since larger, more magnetically complex sunspot groups release more energy and produce more flares. The multipliers are very close to the ones from the Poisson model, so moving to the negative binomial improved how well the model fits the spread of the data without changing the overall conclusions.

## Conclusion

This project modelled the number of C-class solar flares produced by an active region in a 24-hour period, using a count-regression approach based on the region's physical features.

The Zürich class of the sunspot group was the strongest predictor. The flare rate rose steadily with the size and complexity of the region, from about 2.5 times the baseline rate for class C up to roughly 8 to 11 times for classes E and F, while the quietest class sat at or below baseline. Spot distribution added little once the Zürich class was included, and recent activity had a weaker but still detectable effect. These results match the physics, since larger, more magnetically complex regions release more energy and flare more often.

The flare counts were overdispersed, meaning they were more variable than a Poisson model assumes. This was visible in the raw data, confirmed by the Poisson fit's diagnostics, and dealt with by a negative binomial model that fit much better (AIC 1268 vs 1366), so the negative binomial is the more appropriate model here.

There are a few limitations. The predictors are all categorical and partly overlap (one of them, spot size, was dropped because it duplicated information already in the Zürich class), and the dispersion parameter of the negative binomial was fixed at a default rather than estimated from the data. The natural next steps would be to estimate the dispersion directly, to model the rarer M- and X-class flares, and to handle the large number of zero-flare regions properly with a zero-inflated model.